# Ensembl REST

![Ensembl REST](https://proto-bio.github.io/proto-assets/images/tool/ensembl/hero.png)

Five tools share the `ensembl` toolkit, one per endpoint family:

- **`ensembl-lookup`** — gene lookup by Ensembl gene ID **or** gene symbol.
- **`ensembl-sequence`** — DNA / cDNA / CDS / protein sequence retrieval by Ensembl ID.
- **`ensembl-overlap`** — features overlapping a region (genes, exons, regulatory elements, motifs, variants).
- **`ensembl-xrefs`** — cross-references from an Ensembl ID to external databases.
- **`ensembl-vep`** — variant-effect prediction from HGVS notation.

All five call `rest.ensembl.org` (GRCh38) by default; `assembly="GRCh37"` routes to `grch37.rest.ensembl.org`.

In [1]:
from proto_tools.tools.database_retrieval import (
    EnsemblLookupConfig,
    EnsemblLookupInput,
    EnsemblOverlapConfig,
    EnsemblOverlapInput,
    EnsemblSequenceConfig,
    EnsemblSequenceInput,
    EnsemblVEPInput,
    EnsemblXrefsInput,
    run_ensembl_lookup,
    run_ensembl_overlap,
    run_ensembl_sequence,
    run_ensembl_vep,
    run_ensembl_xrefs,
)

## ensembl-lookup — gene lookup with transcript expansion

In [2]:
out = run_ensembl_lookup(EnsemblLookupInput(symbol="BRCA1"), EnsemblLookupConfig(expand=True))
assert out.success
gene = out.result
print(f"{gene.display_name} ({gene.id}): {len(gene.Transcript)} transcripts; canonical={gene.canonical_transcript}")

BRCA1 (ENSG00000012048): 47 transcripts; canonical=ENST00000357654.9


## ensembl-sequence — protein sequence of the canonical transcript

In [3]:
canonical_id = gene.canonical_transcript.split(".")[0]
seq_out = run_ensembl_sequence(
    EnsemblSequenceInput(ensembl_id=canonical_id),
    EnsemblSequenceConfig(sequence_type="protein"),
)
print(f"{seq_out.results[0].id}: {len(seq_out.results[0].seq)} aa")

ENSP00000350283: 1863 aa


## ensembl-overlap — regulatory features at the BRCA1 locus

In [4]:
overlap_out = run_ensembl_overlap(
    EnsemblOverlapInput(ensembl_id=gene.id),
    EnsemblOverlapConfig(overlap_feature="regulatory"),
)
for record in overlap_out.result[:5]:
    print(record.feature_type, record.raw.get("feature_name"))

regulatory None
regulatory None
regulatory None
regulatory None
regulatory None


## ensembl-xrefs — cross-reference Ensembl → UniProt

In [5]:
xrefs_out = run_ensembl_xrefs(EnsemblXrefsInput(ensembl_id=gene.id))
uniprot_id = next(x.primary_id for x in xrefs_out.result if x.dbname == "Uniprot_gn")
print(f"UniProt accession for {gene.display_name}: {uniprot_id}")

UniProt accession for BRCA1: E9PC22


## ensembl-vep — Variant Effect Prediction

HGVS forms supported: genomic (`9:g.22125504G>C`), coding (`ENST00000357654:c.5074G>A`), or protein (`ENSP00000418960:p.Tyr124Cys`).

In [6]:
vep = run_ensembl_vep(EnsemblVEPInput(hgvs="9:g.22125504G>C"))
top = vep.consequences[0]
print(f"{top.allele_string}: {top.most_severe_consequence}, {len(top.transcript_consequences)} transcripts affected")

G/C: intron_variant, 22 transcripts affected


## GRCh37 — same gene, different coordinates

Set `assembly="GRCh37"` for legacy/clinical datasets pinned to the older assembly.

In [7]:
grch37 = run_ensembl_lookup(
    EnsemblLookupInput(ensembl_id=gene.id),
    EnsemblLookupConfig(assembly="GRCh37"),
)
print(f"GRCh38 start: {gene.start} -> GRCh37 start: {grch37.result.start}")

GRCh38 start: 43044292 -> GRCh37 start: 41196312
